In [0]:
import urllib.request

url = "https://data.austintexas.gov/api/views/tyfh-5r8s/rows.csv?accessType=DOWNLOAD"
urllib.request.urlretrieve(url, "/tmp/bikeshare_trips.csv")

print("Descarga completa")

Descarga completa


In [0]:
import pandas as pd

url = "https://data.austintexas.gov/api/views/tyfh-5r8s/rows.csv?accessType=DOWNLOAD"
pdf = pd.read_csv(url)

print(f"Filas descargadas: {len(pdf)}")
pdf.head()

/home/spark-c2164569-fa57-4c3d-845c-f9/.ipykernel/87/command-8972198711414565-3740448556:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  pdf = pd.read_csv(url)


Filas descargadas: 2271153


,Trip ID,Membership or Pass Type,Bicycle ID,Bike Type,Checkout Datetime,Checkout Date,Checkout Time,Checkout Kiosk ID,Checkout Kiosk,Return Kiosk ID,Return Kiosk,Trip Duration Minutes,Month,Year
0,31740060,Local365,003,classic,10/31/2023 11:16:50 PM,10/31/2023,23:16:50,2494.0,2nd/Congress,2504.0,South Congress/Elizabeth,9,10,2023
1,31738417,Local365,21630,electric,10/31/2023 07:18:44 PM,10/31/2023,19:18:44,7187.0,South Congress/Mary,3293.0,East 2nd/Pedernales,22,10,2023
2,31739514,Local365,19434,electric,10/31/2023 09:31:30 PM,10/31/2023,21:31:30,3293.0,East 2nd/Pedernales,4059.0,Nash Hernandez/East @ RBJ South,21,10,2023
3,31739642,Local365,19434,electric,10/31/2023 09:52:56 PM,10/31/2023,21:52:56,4059.0,Nash Hernandez/East @ RBJ South,7187.0,South Congress/Mary,14,10,2023
4,31730597,Local365,17374,electric,10/31/2023 08:47:14 AM,10/31/2023,8:47:14,2495.0,4th/Congress,2561.0,12th/San Jacinto @ State Capitol Visitors Garage,84,10,2023


In [0]:
# Convertir columnas de ID a string para evitar conflictos de tipo
pdf = pdf.astype(str)

df_bronze = spark.createDataFrame(pdf)
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- Trip ID: string (nullable = true)
 |-- Membership or Pass Type: string (nullable = true)
 |-- Bicycle ID: string (nullable = true)
 |-- Bike Type: string (nullable = true)
 |-- Checkout Datetime: string (nullable = true)
 |-- Checkout Date: string (nullable = true)
 |-- Checkout Time: string (nullable = true)
 |-- Checkout Kiosk ID: string (nullable = true)
 |-- Checkout Kiosk: string (nullable = true)
 |-- Return Kiosk ID: string (nullable = true)
 |-- Return Kiosk: string (nullable = true)
 |-- Trip Duration Minutes: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Year: string (nullable = true)

+--------+-----------------------+----------+---------+--------------------+-------------+-------------+-----------------+--------------------+---------------+--------------------+---------------------+-----+----+
| Trip ID|Membership or Pass Type|Bicycle ID|Bike Type|   Checkout Datetime|Checkout Date|Checkout Time|Checkout Kiosk ID|      Checkout Kiosk|Return Ki

In [0]:
df_bronze.columns

['Trip ID',
 'Membership or Pass Type',
 'Bicycle ID',
 'Bike Type',
 'Checkout Datetime',
 'Checkout Date',
 'Checkout Time',
 'Checkout Kiosk ID',
 'Checkout Kiosk',
 'Return Kiosk ID',
 'Return Kiosk',
 'Trip Duration Minutes',
 'Month',
 'Year']

In [0]:
from pyspark.sql.functions import to_timestamp, col, to_date

df_silver = (
    df_bronze
    .withColumn("checkout_ts", to_timestamp(col("Checkout Datetime"), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("trip_duration", col("Trip Duration Minutes").cast("double"))
    .filter(col("checkout_ts").isNotNull())
    .filter(col("Checkout Kiosk").isNotNull())
    .filter(col("trip_duration") > 0)
    .withColumn("fecha", to_date(col("checkout_ts")))
    .select(
        "Trip ID",
        "Membership or Pass Type",
        "Bicycle ID",
        "checkout_ts",
        "Checkout Kiosk",
        "Return Kiosk",
        "trip_duration",
        "fecha"
    )
)

print(f"Filas en Silver: {df_silver.count()}")
df_silver.show(5)

Filas en Silver: 2271153
+--------+-----------------------+----------+-------------------+--------------------+--------------------+-------------+----------+
| Trip ID|Membership or Pass Type|Bicycle ID|        checkout_ts|      Checkout Kiosk|        Return Kiosk|trip_duration|     fecha|
+--------+-----------------------+----------+-------------------+--------------------+--------------------+-------------+----------+
|31740060|               Local365|       003|2023-10-31 23:16:50|        2nd/Congress|South Congress/El...|          9.0|2023-10-31|
|31738417|               Local365|     21630|2023-10-31 19:18:44| South Congress/Mary| East 2nd/Pedernales|         22.0|2023-10-31|
|31739514|               Local365|     19434|2023-10-31 21:31:30| East 2nd/Pedernales|Nash Hernandez/Ea...|         21.0|2023-10-31|
|31739642|               Local365|     19434|2023-10-31 21:52:56|Nash Hernandez/Ea...| South Congress/Mary|         14.0|2023-10-31|
|31730597|               Local365|     17374

In [0]:
df_silver_clean = (
    df_silver
    .withColumnRenamed("Trip ID", "trip_id")
    .withColumnRenamed("Membership or Pass Type", "membership_type")
    .withColumnRenamed("Bicycle ID", "bicycle_id")
    .withColumnRenamed("Checkout Kiosk", "checkout_kiosk")
    .withColumnRenamed("Return Kiosk", "return_kiosk")
)

df_silver_clean.write.mode("overwrite").partitionBy("fecha").format("delta").saveAsTable("silver_bikeshare_trips")

In [0]:
spark.sql("SELECT COUNT(*) AS total FROM silver_bikeshare_trips").show()

+-------+
|  total|
+-------+
|2271153|
+-------+



In [0]:
%sql
DESCRIBE DETAIL silver_bikeshare_trips

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8f841297-53a7-4bb4-b993-2069bed60c24,workspace.default.silver_bikeshare_trips,null,,2026-08-29T23:01:13.131Z,2026-08-29T23:03:19.000Z,List(fecha),List(),3851,44475240,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
EXPLAIN SELECT trip_id, checkout_ts, trip_duration FROM silver_bikeshare_trips WHERE fecha = '2023-10-31'

plan


In [0]:
from pyspark.sql.functions import date_trunc, avg, count, round as spark_round, when

df_gold = (
    df_silver_clean
    .withColumn("month_start", date_trunc("month", col("checkout_ts")))
    .groupBy("checkout_kiosk", "month_start")
    .agg(
        count("*").alias("total_trips"),
        spark_round(avg("trip_duration"), 2).alias("avg_duration_minutes")
    )
)

df_gold.show(10)

+--------------------+-------------------+-----------+--------------------+
|      checkout_kiosk|        month_start|total_trips|avg_duration_minutes|
+--------------------+-------------------+-----------+--------------------+
|            3rd/West|2023-10-01 00:00:00|        582|               24.55|
|Barton Springs/Bo...|2014-10-01 00:00:00|        503|               24.97|
|     Rainey/Driskill|2023-12-01 00:00:00|         82|               31.55|
|Republic Square @...|2016-03-01 00:00:00|        607|               20.36|
|Guadalupe/West Ma...|2015-01-01 00:00:00|        128|               35.29|
|Riverside/South L...|2014-12-01 00:00:00|        340|               31.71|
|      17th/Guadalupe|2014-11-01 00:00:00|        120|               31.63|
|      West & 6th St.|2016-06-01 00:00:00|        285|               21.17|
|East 5th/Broadway...|2014-10-01 00:00:00|        122|               27.69|
|           5th/Bowie|2015-08-01 00:00:00|        664|               16.08|
+-----------

In [0]:
from pyspark.sql.functions import to_date

df_gold_final = df_gold.withColumn("month_start", to_date(col("month_start")))

df_gold_final.write.mode("overwrite").partitionBy("month_start").format("delta").saveAsTable("gold_bikeshare_monthly_summary")

In [0]:
spark.sql("SELECT COUNT(*) AS total FROM gold_bikeshare_monthly_summary").show()

+-----+
|total|
+-----+
| 8262|
+-----+

